# Lab 3 — PCA si K-means pe Ames Housing
## Varianta compacta
Analizam **6 configuratii**: min-max la 98%, 95%, 90% si 80% varianta PCA; L2 si log + min-max doar la 95%. Astfel separam efectul reducerii dimensionalitatii de efectul preprocesarii.

**Rulare:** folosim acelasi fisier `../data/AmesHousing.csv` ca in L1 si L2 si alegem **Run All**. Sunt necesare numpy, pandas, matplotlib, scikit-learn si IPython. Spatiile din numele coloanelor sunt eliminate (`Gr Liv Area` → `GrLivArea`).

**Interpretarile** din celulele Markdown marcate cu *Interpretarea rezultatelor* se bazeaza pe rularea cu `SEED = 42`. Daca modifici preprocesarea, numerele se pot schimba si interpretarile trebuie verificate.

**Encoding:** atributele de calitate (`Po` < `Fa` < `TA` < `Gd` < `Ex`) sunt codificate ordinal 1–5, cu 0 pentru lipsa dotarii; restul variabilelor categorice primesc one-hot. `SalePrice` nu intra in clustering, dar ajuta la interpretare.


## 1. Importuri si configurare
**Scop:** fixam datele de intrare, valorile testate pentru `k` si seed-ul pentru rezultate reproductibile.

`K_OVERRIDES` permite alegerea lui k dupa inspectarea graficului Elbow. Fara o alegere manuala folosim maximul silhouette, ca **propunere**, nu ca dovada a numarului real de grupuri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

SEED = 42
DATA_PATH = "../data/AmesHousing.csv"  # acelasi fisier ca in L1/L2
K_VALUES = list(range(2, 9))
K_OVERRIDES = {}  # exemplu: {("minmax", 0.95): 3}
pd.set_option("display.max_columns", 20)


## 2. Pregatirea datelor si encoding
- **Curatare (ca in L1):** eliminam casele cu `GrLivArea > 4000` si `SalePrice < 300000`.
- Eliminam identificatorii si pretul din atributele de antrenare. `MSSubClass` este tratat ca o categorie.
- **Encoding ordinal** pentru atributele de calitate: 0 = lipsa, 1 = `Po`, ..., 5 = `Ex`.
- Inlocuim valorile numerice lipsa cu mediana si categoriile lipsa cu `Missing`.
- **One-hot encoding** transforma fiecare categorie intr-o coloana 0/1.
- Selectam pentru `log1p` coloane numerice nenegative, asimetrice spre dreapta, cu peste 10 valori distincte. Excludem anii si lunile.

**Interpretare:** logaritmarea comprima valorile mari. Lista rezultata trebuie inspectata: regula statistica este o selectie initiala, nu inlocuieste sensul atributelor. Imputarea generica nu distinge toate cazurile Ames in care lipsa inseamna absenta unei dotari.

In [ ]:
if not Path(DATA_PATH).is_file():
    raise FileNotFoundError(f"Pune CSV-ul langa notebook sau modifica DATA_PATH: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, sep=None, engine="python")
df.columns = df.columns.str.replace(" ", "", regex=False)  # "Gr Liv Area" -> "GrLivArea"
outliers = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
df = df.loc[~outliers].reset_index(drop=True)
print(f"Eliminat {outliers.sum()} outlieri (regula din L1)")
QUALITY = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
           "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
QUALITY_MAP = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
X = df.drop(columns=["Id", "PID", "Order", "SalePrice"], errors="ignore").copy()
for col in QUALITY:
    if col in X:
        X[col] = X[col].map(QUALITY_MAP).fillna(0)  # NaN = dotarea lipseste
if "MSSubClass" in X:
    X["MSSubClass"] = X["MSSubClass"].astype(str)
X = X.dropna(axis=1, how="all")
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()
if not numeric_cols or len(X) <= max(K_VALUES):
    raise ValueError("Sunt necesare coloane numerice si mai mult de 8 case.")
numeric_imputer = SimpleImputer(strategy="median").fit(X[numeric_cols])
categorical_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibilitate cu versiuni mai vechi
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
if categorical_cols:
    encoder.fit(categorical_imputer.fit_transform(X[categorical_cols]))
    categorical_names = encoder.get_feature_names_out(categorical_cols).tolist()
else:
    categorical_names = []
feature_names = numeric_cols + categorical_names
log_cols = [c for c in numeric_cols if X[c].min() >= 0 and X[c].nunique() > 10
            and X[c].skew() > 1 and "year" not in c.lower()
            and c not in ["YrSold", "GarageYrBlt", "MoSold"]]
log_indices = [numeric_cols.index(c) for c in log_cols]

def encode_data(frame, use_log=False):
    frame = frame.reindex(columns=X.columns)
    numeric = numeric_imputer.transform(frame[numeric_cols])
    if use_log:
        if (numeric[:, log_indices] < 0).any():
            raise ValueError("Atributele selectate pentru log trebuie sa fie nenegative.")
        numeric[:, log_indices] = np.log1p(numeric[:, log_indices])
    encoded = (encoder.transform(categorical_imputer.transform(frame[categorical_cols]))
               if categorical_cols else np.empty((len(frame), 0)))
    return np.column_stack([numeric, encoded])

print(f"{len(X)} case; {len(feature_names)} atribute dupa encoding")
print("Atribute logaritmate:", log_cols)


## 3. Preprocesare si PCA
**Min-max** scaleaza separat fiecare coloana in [0, 1]. **L2** imparte toate valorile unei case la lungimea vectorului ei; nu echilibreaza coloanele cu unitati diferite. **Log + min-max** comprima intai atributele selectate, apoi scaleaza coloanele.

**Varianta** masoara imprastierea valorilor. O **componenta principala** este o combinatie liniara a atributelor centrate. PCA ordoneaza componentele dupa varianta explicata.

**Citirea curbei:** pe axa X avem numarul de componente, iar pe Y proportia cumulativa din varianta datelor transformate. O urcare rapida inseamna ca variatia este concentrata in putine directii. Nu inseamna automat informatie mai utila sau clustere mai bune. 95% se refera la varianta reprezentarii respective, nu la 95% din toate informatiile despre case.

In [ ]:
preprocessors = {"minmax": MinMaxScaler(), "L2": Normalizer(norm="l2"),
                 "log_minmax": MinMaxScaler()}
representations = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, scaler) in zip(axes, preprocessors.items()):
    scaled = scaler.fit_transform(encode_data(X, name == "log_minmax"))
    pca = PCA(svd_solver="full")
    scores = pca.fit_transform(scaled)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    representations[name] = dict(scaler=scaler, pca=pca, scores=scores, cumulative=cumulative)
    ax.plot(np.arange(1, len(cumulative)+1), cumulative)
    for threshold in [.80, .90, .95, .98]:
        ax.axhline(threshold, linestyle="--", alpha=.3)
    ax.set(title=name, xlabel="Numar de componente", ylabel="Varianta cumulativa")
    ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
variance_table = pd.DataFrame({name: pd.Series(r["pca"].explained_variance_ratio_)
                              for name, r in representations.items()})
variance_table.index = [f"PC{i+1}" for i in range(len(variance_table))]
display(variance_table.head(10).round(4))
print("Toate proportiile sunt disponibile in variance_table.")


### Interpretarea rezultatelor

- **Min-max:** PC1 explica doar **12.2%** din varianta, PC2 **7.3%**, PC3 **4.5%**; primele 10 componente impreuna explica aproximativ **43%**. Varianta este imprastiata pe multe directii, de aceea curba urca lent. Din cele **288 de atribute** dupa encoding raman **135** componente pentru 98%, **98** pentru 95%, **72** pentru 90% si **45** pentru 80%. Chiar si la 80% avem nevoie de 45 de dimensiuni: datele Ames nu au o structura „joasa” evidenta.
- **De ce atat de multe componente la min-max?** Aproximativ 250 din cele 288 de coloane sunt coloane one-hot 0/1. Dupa min-max, o coloana binara poate avea varianta pana la 0.25, in timp ce atributele continue asimetrice (de ex. `LotArea`) au majoritatea valorilor comprimate langa 0 si deci varianta mica. Varianta totala este astfel dominata de multe coloane categorice slab corelate intre ele, fiecare adaugand putin.
- **L2:** PC1 explica singura **74.6%**, iar **2** componente ajung pentru 80%, **5** pentru 95% si **8** pentru 98%. Aceasta concentrare nu inseamna ca L2 a gasit o structura mai buna: `Normalizer` lucreaza pe randuri si nu egalizeaza coloanele. Vectorul fiecarei case este dominat de valorile numeric mari (`LotArea` ~10 000, anii ~2000, suprafetele ~1000), deci aproape toata variatia este data de raportul dintre suprafata terenului si ceilalti termeni mari. Coloanele one-hot (0/1) devin practic neglijabile.
- **Log + min-max:** curba este aproape identica cu min-max (**136 / 100 / 74 / 46** componente). Logaritmarea a modificat doar 12 coloane, majoritatea rare si cu multe zerouri (porch-uri, piscina, `MiscVal`), iar `GrLivArea` nu a fost selectata (asimetria ei este sub 1 dupa eliminarea outlierilor). Efectul asupra variantei totale, dominate de one-hot, este minor.

## 4. Antrenarea K-means si tabelul comparativ
**K-means** atribuie casele unor centre si muta centrele spre media caselor atribuite. Minimizeaza suma distantelor patratice fata de centre: **inertia**.

Testam `k=2,...,8` pentru cele sase configuratii. **Silhouette** compara distanta medie fata de propriul grup cu cea fata de cel mai apropiat alt grup: aproape de 1 inseamna separare buna, aproape de 0 separare slaba, iar valorile negative sugereaza o atribuire nepotrivita.

Calculam scorurile in toate componentele pastrate. Aceasta este analiza exploratorie pe setul disponibil, nu evaluare predictiva pe un set independent.

In [ ]:
configurations = [("minmax", t) for t in [.98, .95, .90, .80]] + [("L2", .95), ("log_minmax", .95)]
results, rows = {}, []
for name, threshold in configurations:
    rep = representations[name]
    count = min(len(rep["cumulative"]), int(np.searchsorted(rep["cumulative"], threshold)+1))
    Z = rep["scores"][:, :count]
    models, inertias, silhouettes = {}, [], []
    one = KMeans(n_clusters=1, n_init=10, random_state=SEED).fit(Z)
    for k in K_VALUES:
        model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Z)
        models[k] = model
        inertias.append(model.inertia_)
        silhouettes.append(silhouette_score(Z, model.labels_))
    suggested = K_VALUES[int(np.argmax(silhouettes))]
    chosen = K_OVERRIDES.get((name, threshold), suggested)
    if chosen not in models:
        raise ValueError("Alege k intre 2 si 8 in K_OVERRIDES.")
    results[(name, threshold)] = dict(Z=Z, n_components=count, models=models,
        inertias=[one.inertia_]+inertias, silhouettes=silhouettes,
        k=chosen, model=models[chosen], labels=models[chosen].labels_)
    rows.append({"Metoda": name, "Prag": threshold, "Componente": count,
                 "Varianta": rep["cumulative"][count-1], "k": chosen,
                 "Silhouette": silhouettes[K_VALUES.index(chosen)]})
summary = pd.DataFrame(rows)
display(summary.round(4))
for row in rows:
    display(Markdown(f"- **{row['Metoda']}, {row['Prag']:.0%}:** {row['Componente']} componente, "
                     f"k={row['k']}, silhouette={row['Silhouette']:.3f}."))


### Interpretarea rezultatelor

- **Min-max, 98% / 95% / 90%:** toate trei aleg **k=3**, cu silhouette **0.094 / 0.097 / 0.102**. Dimensiunile clusterelor sunt practic aceleasi (~1035 / ~800 / ~1090 case), doar numerotarea difera. Reducerea de la 135 la 72 de componente nu schimba impartirea, deci componentele eliminate contin in principal variatie mica, de tip zgomot.
- **Min-max, 80%:** se trece la **k=2** (silhouette 0.116), dar diferenta fata de k=3 (0.114) este neglijabila. Cu 45 de componente, doua dintre cele trei grupuri (casele vechi si cele din anii '60) devin mai greu de separat si sunt unite.
- **Silhouette creste usor cand scade pragul** (0.094 → 0.116). In spatii cu mai putine dimensiuni distantele sunt mai putin „diluate” de componente cu varianta mica, dar aceasta nu dovedeste ca s-a pastrat mai multa informatie utila.
- **Valorile absolute sunt mici (~0.1):** clusterele min-max se suprapun mult; casele formeaza mai degraba un continuu decat grupuri clar delimitate.
- **L2, 95%:** silhouette **0.624** cu **k=2**, de sase ori mai mare. Separarea este insa intre 2583 si 344 de case si se bazeaza aproape doar pe `LotArea` (vezi sectiunile 8 si 9). Scorul mare reflecta o separare geometrica usoara intr-un spatiu aproape unidimensional, nu grupuri mai utile.
- **Log + min-max, 95%:** **k=3**, silhouette **0.096**, aproape identic cu min-max (0.097). Logaritmarea nu schimba practic rezultatul.

## 5. Elbow si silhouette mediu
**Elbow:** cautam un cot dupa care scaderea inertiei devine mai lenta. Inertia scade firesc cand creste k; minimul ei nu este criteriul de alegere.

**Silhouette mediu:** cautam scoruri ridicate, dar diferentele foarte mici nu sustin o alegere categorica. Linia rosie arata k ales, initial dupa silhouette.

**Decizia** pentru k este explicata in interpretarea de dupa grafic. Pentru o alegere diferita modifica `K_OVERRIDES` si reruleaza de la celula de antrenare.

Nu comparam direct inertia intre min-max si L2: distantele au alte scari.

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(12, 18))
for row, key in enumerate(configurations):
    r = results[key]; title = f"{key[0]}, PCA {key[1]:.0%}"
    axes[row, 0].plot([1]+K_VALUES, r["inertias"], marker="o")
    axes[row, 0].set(title=title+" — Elbow", ylabel="Inertia")
    axes[row, 1].plot(K_VALUES, r["silhouettes"], marker="o")
    axes[row, 1].axvline(r["k"], color="red", linestyle="--")
    axes[row, 1].set(title=title+" — Silhouette", ylabel="Scor mediu")
    for ax in axes[row]:
        ax.set_xlabel("k"); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

- **Elbow la min-max (toate pragurile):** curba este neteda, fara un cot clar. Cea mai mare scadere apare de la k=1 la k=2 (≈ −3800 la 95%), urmata de k=2→3 (≈ −2000) si k=3→4 (≈ −1200); dupa k=4 scaderile devin mici si aproape constante (≈ −500…−700). Cotul, cat se observa, se afla in zona **k=3–4**.
- **Silhouette la min-max:** maximul este la **k=3** pentru 98%, 95% si 90%, cu k=2 aproape la egalitate, iar la **k=4** scorul scade brusc (de ex. 0.097 → 0.077 la 95%). Combinand cele doua criterii, **k=3** este o alegere rezonabila: este in zona cotului si are cel mai bun silhouette. La 80%, k=2 si k=3 sunt practic egale.
- **L2:** aici exista un cot foarte clar la **k=2** (inertia scade de la 134 la 63, apoi doar la 46), iar silhouette scade de la 0.62 la 0.29 la k=3. Ambele criterii indica k=2, deoarece reprezentarea L2 contine in esenta o singura directie dominanta (o masa mare de case si o „coada” de case cu teren mic).
- **Log + min-max:** curbe practic identice cu min-max; aceeasi alegere, k=3.
- Valorile inertiei nu se compara intre min-max (~30 000) si L2 (~30–130): scara distantelor este complet diferita (vectorii L2 au lungimea 1).

## 6. Diagrame silhouette detaliate
Pentru **min-max la 95%**, comparam k=2, 3 si 4. Fiecare banda colorata este un cluster; fiecare linie orizontala foarte subtire este o casa. Inaltimea benzii indica numarul de case, iar extinderea spre dreapta indica scorul. Casele sunt sortate dupa scor in fiecare cluster. Linia rosie este media.

**Interpretare:** benzi extinse spre dreapta indica separare buna. Portiunile negative indica case mai apropiate, in medie, de alt grup. Grupurile de dimensiuni diferite nu sunt automat gresite. Nu repetam aceste figuri pentru toate configuratiile; comparatia generala ramane in tabel si curbele medii.

In [ ]:
baseline_key = ("minmax", .95)
baseline = results[baseline_key]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, k in zip(axes, [2, 3, 4]):
    labels = baseline["models"][k].labels_
    values = silhouette_samples(baseline["Z"], labels)
    lower = 10
    for cluster in range(k):
        v = np.sort(values[labels == cluster]); upper = lower + len(v)
        ax.fill_betweenx(np.arange(lower, upper), 0, v, alpha=.8)
        ax.text(-.09, (lower+upper)/2, str(cluster)); lower = upper+10
    ax.axvline(values.mean(), color="red", linestyle="--")
    ax.axvline(0, color="black", linewidth=.7)
    ax.set(title=f"k={k}, media={values.mean():.3f}", xlabel="Silhouette", ylabel="Case")
    ax.set_yticks([])
plt.suptitle("Min-max, PCA 95%")
plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

- **k=2 (media 0.096):** clusterul 0 are scoruri pozitive pana la ~0.27, dar clusterul 1 are o portiune vizibila de valori negative: multe case sunt la granita si ar putea apartine celuilalt grup.
- **k=3 (media 0.097):** clusterul **1** (casele noi, de calitate ridicata, vezi sectiunea 9) este cel mai bine separat, cu scoruri de pana la ~0.24 si aproape fara valori negative. Clusterele 0 (case vechi) si 2 (case din anii '60) au scoruri mai mici si mici cozi negative: sunt mai apropiate intre ele decat de casele noi. De aceea, la pragul de 80%, ele sunt unite intr-un singur cluster.
- **k=4 (media 0.077):** apar mai multe portiuni negative, iar clusterul suplimentar (3) are scoruri mici: impartirea unui grup existent in doua nu creeaza un grup nou bine separat.
- **Concluzie:** k=3 are distributia cea mai echilibrata, cu putine atribuiri gresite, dar toate scorurile individuale raman sub ~0.27. Grupurile sunt reale ca tendinte, nu ca insule separate.

## 7. Vizualizarea clusterelor in 2D
Fiecare punct este o casa; culoarea indica clusterul, iar **X** marcheaza centroidul proiectat. Axele sunt PC1 si PC2, nu suprafata si pretul.

**Interpretare:** K-means foloseste toate componentele retinute, desenul doar doua. Apropierea sau separarea din imagine poate ascunde diferente pe celelalte componente. Intre pragurile aceleiasi preprocesari pozitiile punctelor raman aceleasi, dar etichetele se pot schimba. Culorile si numerele clusterelor nu au aceeasi semnificatie intre antrenari.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, key in zip(axes.flat, configurations):
    r = results[key]
    if r["n_components"] < 2:
        ax.text(.1, .5, "O singura componenta retinuta; grafic 2D indisponibil.")
        continue
    Z = r["Z"]; centers = r["model"].cluster_centers_
    ax.scatter(Z[:, 0], Z[:, 1], c=r["labels"], cmap="tab10", s=12, alpha=.55)
    ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=150, label="Centroizi")
    ax.set(title=f"{key[0]}, {key[1]:.0%}: {r['n_components']} PCs, k={r['k']}", xlabel="PC1", ylabel="PC2")
    ax.legend()
plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

- **Min-max 98% / 95% / 90%:** imaginea este aceeasi: trei regiuni, una in dreapta (PC1 > ~0.5), una sus-stanga si una jos-stanga. Culorile se schimba doar pentru ca etichetele sunt atribuite arbitrar; centroizii sunt in aceleasi pozitii (≈(1.4, −0.4), (−0.4, 1.0), (−1.4, −0.8)). Deci **nu exista diferente reale intre aceste trei praguri**.
- **Min-max 80% (k=2):** separarea se face aproape doar dupa PC1: partea dreapta (case moderne) contra restului. Cele doua grupuri din stanga sunt unite.
- **Ce inseamna axele (vezi sectiunea 8):** PC1 mare = constructie moderna (fundatie din beton turnat, siding vinil, garaj atasat); PC2 mare = case cu un singur nivel, fundatie din blocuri de beton. Clusterul din dreapta contine casele noi, cel sus-stanga casele cu un nivel din anii '50–'70, iar cel jos-stanga casele vechi.
- **Suprapunere:** granitele dintre culori sunt continue, fara spatii goale. Aceasta confirma silhouette-ul mic: K-means taie un nor continuu in regiuni, nu gaseste clustere naturale separate.
- **L2:** axele au alta scara (±0.4). Exista o masa densa in dreapta si o „coada” rarefiata in stanga (PC1 negativ). Clusterul mic (344 case) este exact aceasta coada, adica locuintele cu teren foarte mic. Separarea vizuala este clara, dar descrie aproape un singur atribut.
- **Log + min-max:** aceeasi structura in trei regiuni ca la min-max (eventual cu o axa oglindita), deci logaritmarea nu schimba gruparea.

## 8. Influenta atributelor: loadings
Aplicam formula ceruta: `loadings = pca.components_.T * np.sqrt(pca.explained_variance_)`. Afisam cele mai mari opt contributii in valoare absoluta pentru PC1 si PC2.

**Interpretare:** atributele cu magnitudine mare caracterizeaza componenta. Semnul arata sensul asocierii cu axa, nu o influenta buna sau rea. Semnul unei componente se poate inversa fara sa schimbe PCA. Loadings nu arata cauzalitate si nu sunt automat corelatii pentru datele noastre nestandardizate.

Daca la L2 domina cateva suprafete sau ani, aceasta sustine explicatia ca scala atributelor concentreaza variatia. Nu atribuim componentei numele „marime” sau „calitate” fara sa inspectam atributele dominante.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 13))
for row, (name, rep) in enumerate(representations.items()):
    pca = rep["pca"]
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    for component in range(2):
        contributions = pd.Series(loadings[:, component], index=feature_names)
        selected = contributions.loc[contributions.abs().nlargest(8).index].sort_values()
        axes[row, component].barh(selected.index, selected.values,
            color=["steelblue" if v >= 0 else "tomato" for v in selected])
        axes[row, component].axvline(0, color="black", linewidth=.7)
        axes[row, component].set(title=f"{name} — PC{component+1}", xlabel="Loading")
        top = contributions.abs().nlargest(3).index.tolist()
        display(Markdown(f"**{name}, PC{component+1}:** atribute dominante: {', '.join(top)}."))
plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

- **Min-max, PC1 („modern vs. vechi”):** loadings pozitive mari pentru `Foundation_PConc` (~0.37), `Exterior1st/2nd_VinylSd`, `GarageType_Attchd`, `BsmtFinType1_GLQ`, `YearRemod/Add`; negative pentru `GarageFinish_Unf` si `GarageType_Detchd`. Casele cu PC1 mare sunt construite sau renovate recent, cu materiale moderne si garaj atasat; cele cu PC1 mic sunt vechi, cu garaj detasat si nefinisat.
- **Min-max, PC2 („un nivel vs. doua niveluri”):** pozitive pentru `HouseStyle_1Story`, `MSSubClass_20`, `Foundation_CBlock`, `MSZoning_RL`; negative pentru `HouseStyle_2Story`, `BsmtFinType1_Unf`, `Foundation_PConc`. PC2 separa casele tip ranch cu un nivel, din anii '50–'70, de casele cu doua niveluri.
- **Dominanta one-hot:** la min-max, componentele principale sunt descrise aproape numai de coloane categorice (stil, fundatie, garaj, exterior). Coloanele binare au varianta mare dupa min-max, deci PCA le acorda importanta. Prin urmare, clusterele min-max reflecta mai ales **tipul si epoca constructiei**, nu suprafata.
- **L2, PC1:** `LotArea` are singura loading pozitiv, iar anii (`YrSold`, `YearBuilt`, `GarageYrBlt`, `YearRemod/Add`) si suprafetele au loadings negative. Dupa normalizarea randului, aceasta axa masoara cat din „lungimea” vectorului unei case provine din teren fata de ani: practic, **marimea relativa a terenului**. PC2 opune subsolul si parterul (`TotalBsmtSF`, `BsmtFinSF1`, `1stFlrSF`) etajului 2 (`2ndFlrSF`). Loadings sunt mici (≤0.12), deoarece vectorii au lungimea 1.
- **Log + min-max:** aceleasi atribute dominante ca la min-max, in aproape aceeasi ordine (apare in plus `MasVnrType_Missing`). Confirma ca logaritmarea a schimbat foarte putin reprezentarea.
- Semnul global al unei componente este arbitrar; conteaza ca atributele cu semne opuse se afla la capete opuse ale aceleiasi axe.

## 9. Caracterizarea clusterelor
Comparam **medianele atributelor originale**, numarul de case si cartierul dominant, la 95% PCA pentru toate cele trei metode. Boxploturile sunt doar pentru min-max.

**Citirea boxplotului:** linia din cutie este mediana, cutia contine jumatatea centrala a valorilor. Ascundem punctele extreme doar in desen; ele raman in analiza.

**Interpretare:** un cluster cu suprafata si calitate peste medianele generale poate fi descris ca avand case mai mari si mai calitative relativ la acest set. Anul mai mare inseamna case mai noi. Pretul este doar o verificare descriptiva, nu a fost folosit la grupare. Rezumatele de mai jos raporteaza diferentele reale; separarea geometrica nu garanteaza utilitatea practica.

In [ ]:
profile_cols = [c for c in ["GrLivArea", "LotArea", "OverallQual", "YearBuilt", "GarageCars", "SalePrice"] if c in df.columns]
comparison_keys = [("minmax", .95), ("L2", .95), ("log_minmax", .95)]
profiles = {}
for key in comparison_keys:
    labeled = df.assign(Cluster=results[key]["labels"])
    grouped = labeled.groupby("Cluster")
    profile = grouped[profile_cols].median()
    profiles[key] = profile.copy()
    profile.insert(0, "Numar case", grouped.size())
    if "Neighborhood" in df:
        profile["Cartier dominant"] = grouped["Neighborhood"].agg(
            lambda s: s.mode().iloc[0] if not s.mode().empty else "Necunoscut")
    display(Markdown(f"### {key[0]}, PCA 95%")); display(profile.round(1))
    for cluster, values in profile.iterrows():
        descriptions = []
        for col in profile_cols:
            ref = df[col].median(); value = values[col]
            direction = "peste" if value > ref else "sub" if value < ref else "egala cu"
            descriptions.append(f"{col}: {value:.1f} ({direction} mediana generala {ref:.1f})")
        display(Markdown(f"- **Cluster {cluster}**, {int(values['Numar case'])} case: " + "; ".join(descriptions) + "."))
labeled = df.assign(Cluster=baseline["labels"])
plot_cols = [c for c in ["GrLivArea", "OverallQual", "SalePrice"] if c in df]
if plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(5*len(plot_cols), 4), squeeze=False)
    ids = sorted(labeled.Cluster.unique())
    for ax, col in zip(axes.flat, plot_cols):
        ax.boxplot([labeled.loc[labeled.Cluster == k, col].dropna() for k in ids], showfliers=False)
        ax.set_xticks(range(1, len(ids)+1), labels=ids)
        ax.set(title=col, xlabel="Cluster")
    plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

**Min-max, PCA 95% (k=3):**
- **Cluster 0 — case vechi, mici, ieftine (802 case):** construite in jurul anului **1932**, teren mic (**7500 sqft**), calitate **5**, garaj pentru **1** masina, pret median **\$123 250**, cartier dominant **OldTown**. In clusterul 0 gasim tendential case vechi din zona centrala istorica, pe loturi inguste.
- **Cluster 1 — case noi, mai mari, de calitate (1089 case):** an median **2003**, **1665 sqft**, calitate **7**, garaj pentru 2 masini, pret median **\$222 000**, cartier dominant **CollgCr**. Sunt casele moderne din cartierele noi, peste mediana la aproape toate atributele.
- **Cluster 2 — case cu un nivel din anii '60 (1036 case):** an median **1966**, cea mai mica suprafata locuibila (**1224 sqft**), dar teren peste mediana (**9894 sqft**), calitate **5**, garaj pentru 2 masini, pret **\$146 500**, cartier dominant **NAmes**. Corespund caselor tip ranch suburbane (PC2 mare: `1Story`, `CBlock`).

**Boxploturi:** clusterul 1 se distinge clar la calitate (cutia 7–8) si pret (de la ~\$85 000 la peste \$400 000, deci si cel mai eterogen). Clusterele 0 si 2 se suprapun mult la suprafata, calitate si pret: diferenta dintre ele este data in principal de **vechime, stil si teren**, nu de marime. Pretul nu a fost folosit la antrenare, dar ordinea \$123k < \$146k < \$222k arata ca gruparea surprinde o structura asociata cu valoarea casei.

**L2, PCA 95% (k=2):** un grup mare (2583 case) apropiat de mediana generala si un grup mic (344 case) cu teren foarte mic (**3632 sqft**), mai nou (1984), cartier dominant **Somerst**: locuinte tip townhouse sau loturi mici. Calitatea (6) si garajul sunt egale in cele doua grupuri, iar pretul difera putin. Gruparea L2 este practic o impartire dupa `LotArea`.

**Log + min-max, PCA 95% (k=3):** aceleasi trei profiluri ca la min-max (2003/CollgCr, 1966/NAmes, 1934/OldTown) si aproape aceleasi dimensiuni (1086 / 1025 / 816). Logaritmarea a mutat doar cateva case intre clustere.

## 10. Inferenta pentru case noi
Inventam **patru case plauzibile**. Pornim de la o casa „tipica” (mediana pentru atributele numerice, categoria cea mai frecventa pentru cele categorice) si setam explicit atributele importante: suprafata, calitatea, anul, garajul, cartierul etc. Exemplele **nu sunt un set de test independent**, ci o verificare intuitiva.

| Casa | Descriere |
|---|---|
| Casa mica veche | 1925, OldTown, 850 sqft, calitate 4, garaj detasat pentru o masina, fara aer conditionat central |
| Casa familiala tipica | 1998, Gilbert, 2 etaje, 1700 sqft, calitate 6, garaj atasat pentru 2 masini |
| Casa noua de lux | 2008, NridgHt, 2 etaje, 3000 sqft, calitate 9, finisaje excelente, garaj pentru 3 masini |
| Casa mare veche | 1915, OldTown, 2 etaje, 2400 sqft, calitate 5 — caz atipic: mare, dar veche |

Aplicam in aceeasi ordine **imputare → encoding → min-max → PCA → predict**. Folosim exclusiv transformarile deja invatate (`transform`), nu `fit` pe casele noi.

**De ce PCA?** Centrele K-means exista in spatiul componentelor principale. Datele noi trebuie aduse in acelasi spatiu, cu aceleasi axe si aceeasi centrare.

**Interpretare:** comparam casa cu mediana clusterului atribuit. Atribuirea se bazeaza pe toate atributele transformate, deci o diferenta pe o singura coloana nu indica automat o eroare. O casa va fi atribuita unui cluster chiar daca este atipica; `predict` nu confirma ca exemplul este realist.

In [ ]:
if not {"GrLivArea", "OverallQual"}.issubset(X.columns):
    raise ValueError("Exemplele necesita GrLivArea si OverallQual; adapteaza numele la CSV.")
template = pd.Series({c: X[c].median() for c in numeric_cols} |
                     {c: X[c].mode().iloc[0] for c in categorical_cols})
specs = {
    "Casa mica veche": dict(MSSubClass="30", Neighborhood="OldTown", HouseStyle="1Story", LotArea=6000,
        OverallQual=4, OverallCond=5, YearBuilt=1925, **{"YearRemod/Add": 1950}, Foundation="BrkTil",
        ExterQual=3, KitchenQual=2, BsmtQual=3, HeatingQC=3, CentralAir="N", TotalBsmtSF=700,
        BsmtFinSF1=0, BsmtUnfSF=700, **{"1stFlrSF": 850, "2ndFlrSF": 0}, GrLivArea=850, FullBath=1,
        HalfBath=0, BedroomAbvGr=2, TotRmsAbvGrd=4, Fireplaces=0, FireplaceQu=0, GarageType="Detchd",
        GarageYrBlt=1925, GarageFinish="Unf", GarageCars=1, GarageArea=240),
    "Casa familiala tipica": dict(MSSubClass="60", Neighborhood="Gilbert", HouseStyle="2Story", LotArea=9500,
        OverallQual=6, OverallCond=5, YearBuilt=1998, **{"YearRemod/Add": 1998}, Foundation="PConc",
        ExterQual=3, KitchenQual=3, BsmtQual=4, HeatingQC=5, CentralAir="Y", TotalBsmtSF=850,
        BsmtFinSF1=300, BsmtUnfSF=550, **{"1stFlrSF": 900, "2ndFlrSF": 800}, GrLivArea=1700, FullBath=2,
        HalfBath=1, BedroomAbvGr=3, TotRmsAbvGrd=7, Fireplaces=1, FireplaceQu=3, GarageType="Attchd",
        GarageYrBlt=1998, GarageFinish="RFn", GarageCars=2, GarageArea=450),
    "Casa noua de lux": dict(MSSubClass="60", Neighborhood="NridgHt", HouseStyle="2Story", LotArea=13000,
        OverallQual=9, OverallCond=5, YearBuilt=2008, **{"YearRemod/Add": 2008}, Foundation="PConc",
        ExterQual=5, KitchenQual=5, BsmtQual=5, HeatingQC=5, CentralAir="Y", TotalBsmtSF=1600,
        BsmtFinSF1=1200, BsmtUnfSF=400, **{"1stFlrSF": 1600, "2ndFlrSF": 1400}, GrLivArea=3000, FullBath=3,
        HalfBath=1, BedroomAbvGr=4, TotRmsAbvGrd=10, Fireplaces=2, FireplaceQu=4, GarageType="BuiltIn",
        GarageYrBlt=2008, GarageFinish="Fin", GarageCars=3, GarageArea=800, MasVnrArea=300),
    "Casa mare veche": dict(MSSubClass="70", Neighborhood="OldTown", HouseStyle="2Story", LotArea=10000,
        OverallQual=5, OverallCond=6, YearBuilt=1915, **{"YearRemod/Add": 1980}, Foundation="BrkTil",
        ExterQual=3, KitchenQual=3, BsmtQual=2, HeatingQC=3, CentralAir="Y", TotalBsmtSF=1100,
        BsmtFinSF1=0, BsmtUnfSF=1100, **{"1stFlrSF": 1250, "2ndFlrSF": 1150}, GrLivArea=2400, FullBath=2,
        HalfBath=0, BedroomAbvGr=4, TotRmsAbvGrd=9, Fireplaces=1, FireplaceQu=3, GarageType="Detchd",
        GarageYrBlt=1950, GarageFinish="Unf", GarageCars=2, GarageArea=400),
}
new_houses = pd.DataFrame([template.to_dict() | {k: v for k, v in s.items() if k in X} for s in specs.values()],
                          index=list(specs))
rep = representations["minmax"]
new_scaled = rep["scaler"].transform(encode_data(new_houses))
new_Z = rep["pca"].transform(new_scaled)[:, :baseline["n_components"]]
predictions = baseline["model"].predict(new_Z)
shown = [c for c in profile_cols if c in new_houses]
display(new_houses[shown].assign(Cluster=predictions))
for house, cluster in zip(new_houses.index, predictions):
    display(Markdown(f"**{house} → cluster {cluster}.** Comparam valorile originale cu mediana grupului:"))
    display(pd.DataFrame({"Casa noua": new_houses.loc[house, shown],
                          "Mediana clusterului": profiles[baseline_key].loc[cluster, shown]}))
if baseline["n_components"] >= 2:
    Z = baseline["Z"]; centers = baseline["model"].cluster_centers_
    plt.figure(figsize=(8, 6))
    plt.scatter(Z[:, 0], Z[:, 1], c=baseline["labels"], cmap="tab10", s=10, alpha=.25)
    plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=150, label="Centroizi")
    plt.scatter(new_Z[:, 0], new_Z[:, 1], c="red", marker="*", s=300, edgecolor="black", label="Case noi")
    for (x, y), name in zip(new_Z[:, :2], new_houses.index):
        plt.annotate(name, (x, y), xytext=(6, 6), textcoords="offset points")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Casele inventate in spatiul PCA (min-max, 95%)")
    plt.legend(); plt.tight_layout(); plt.show()


### Interpretarea rezultatelor

| Casa | Cluster | Profilul clusterului | Corect intuitiv? |
|---|---|---|---|
| Casa mica veche (1925, OldTown, 850 sqft, Q4) | **0** | case vechi, OldTown, 1932, Q5 | **Da.** Se potriveste la toate atributele: an, cartier, calitate, garaj pentru o masina. |
| Casa familiala tipica (1998, Gilbert, 1700 sqft, Q6) | **1** | case noi, 2003, 1665 sqft, Q7 | **Da.** Suprafata si anul sunt foarte apropiate de mediana; calitatea 6 este putin sub 7, dar fundatia `PConc` si garajul atasat sunt tipice clusterului. |
| Casa noua de lux (2008, NridgHt, 3000 sqft, Q9) | **1** | case noi, 2003, Q7 | **Da, cu observatie.** Cu k=3 nu exista un cluster separat pentru case de lux, deci este pusa in grupul caselor moderne. In grafic se afla la marginea clusterului (PC1 mare): este un caz extrem al aceluiasi tip. |
| Casa mare veche (1915, OldTown, 2400 sqft, Q5) | **0** | case vechi, 1321 sqft | **Partial.** Dupa vechime, cartier, fundatie `BrkTil` si garaj detasat atribuirea este corecta, desi suprafata este aproape dubla fata de mediana clusterului. In grafic se afla aproape de granita cu clusterul 2. |

**Ce arata exemplele:** modelul grupeaza casele mai ales dupa **epoca si tipul constructiei**, nu dupa marime. Un atribut foarte diferit (suprafata casei mari vechi) nu schimba atribuirea, pentru ca distanta se calculeaza pe toate cele 288 de atribute transformate, dominate de coloanele categorice. Aceasta este consistenta cu loadings din sectiunea 8.

**De ce PCA inainte de `predict`:** centroizii au 98 de coordonate in spatiul componentelor principale (min-max, 95%). O casa noua trebuie trecuta prin **aceleasi** transformari invatate pe datele de antrenare (imputare → encoding → min-max → PCA, toate cu `transform`) ca sa fie descrisa pe aceleasi axe, cu aceeasi centrare si scalare. Un vector brut sau unul transformat cu un `fit` nou ar avea alt numar de dimensiuni sau alt sistem de coordonate, iar distanta fata de centroizi nu ar avea sens.

## 11. Concluzii si limite

### Concluzii

1. **Dimensionalitate (min-max):** din 288 de atribute dupa encoding, PCA pastreaza **135 / 98 / 72 / 45** componente pentru **98% / 95% / 90% / 80%** din varianta. PC1 explica doar 12.2%. Varianta este imprastiata, deoarece reprezentarea este dominata de multe coloane one-hot slab corelate. Se poate reduce dimensiunea cu ~50–65% fara pierderi mari, dar nu exista cateva componente care sa rezume casele.
2. **Pragul PCA aproape nu conteaza pentru clustering:** la 98%, 95% si 90% se obtine aceeasi impartire in 3 grupuri (dimensiuni ~800 / ~1035 / ~1090). Abia la 80% doua grupuri se unesc (k=2). Componentele eliminate contin in principal variatie mica, deci pentru K-means un prag de **90%** (72 de componente) este suficient si mai ieftin computational.
3. **Numarul de clustere:** curba elbow este neteda, cu un cot slab la k=3–4; silhouette are maximul la **k=3**, iar la k=4 scade brusc. Alegem **k=3** pentru min-max, pe baza ambelor criterii si a profilurilor clare.
4. **Cele trei tipuri de case (min-max, k=3):**
   - **case vechi** (~1932, OldTown, teren mic, calitate 5, ~\$123k);
   - **case cu un nivel din anii '60** (~1966, NAmes, suprafata mica, teren mare, ~\$146k);
   - **case moderne** (~2003, CollgCr, 1665 sqft, calitate 7, ~\$222k).
   Pretul nu a participat la antrenare, dar este ordonat clar intre grupuri: clusteringul surprinde o structura legata de valoarea casei.
5. **Ce descriu componentele:** PC1 la min-max este o axa **„modern vs. vechi”** (fundatie beton turnat, siding vinil, garaj atasat), iar PC2 este o axa **„un nivel vs. doua niveluri”**. Clusterele sunt deci definite mai ales de **epoca si stilul constructiei**, nu de suprafata, lucru confirmat si de casa mare veche, atribuita grupului caselor vechi.
6. **Min-max vs. normalizare L2:** L2 obtine un silhouette mult mai mare (**0.62** fata de **0.10**), dar impartirea este 2583 vs. 344 de case si separa practic doar casele cu **teren foarte mic** (`LotArea` ~3600). `Normalizer` lucreaza pe randuri si nu egalizeaza scalele coloanelor; vectorul este dominat de `LotArea` si de ani, iar atributele categorice dispar. Un silhouette mare **nu inseamna** grupuri mai utile: pentru interpretare, **min-max este reprezentarea mai potrivita**.
7. **Logaritmarea:** cu min-max, `log1p` nu schimba aproape nimic (aceleasi 3 profiluri, silhouette 0.096 vs. 0.097). Au fost logaritmate doar 12 coloane, majoritatea rare (porch, piscina, `MiscVal`), iar spatiul min-max este dominat de coloanele one-hot. Logaritmarea ar avea efect mai mare intr-o reprezentare dominata de atributele numerice mari, cum este L2.
8. **Case noi:** toate cele patru case inventate primesc clustere plauzibile. Casa de lux ajunge in grupul caselor moderne pentru ca nu exista un grup separat de lux la k=3. Casa mare veche arata ca modelul prioritizeaza vechimea si stilul fata de marime.

### Limite

- **Silhouette mic (~0.1) la min-max:** clusterele se suprapun mult. Casele Ames formeaza un continuu, iar K-means il imparte in regiuni; grupurile sunt tendinte, nu categorii naturale clar separate.
- **K-means presupune clustere aproximativ sferice si de marime similara** si foloseste distanta euclidiana. Pe date cu multe coloane binare, aceasta distanta este influentata puternic de encoding. Metode ca GMM, DBSCAN sau o distanta potrivita pentru date mixte (de ex. Gower) ar putea da alte rezultate.
- **Ponderea one-hot:** variabilele categorice cu multe niveluri (cartier, exterior, `MSSubClass`) produc multe coloane si cantaresc mai mult in distanta decat un atribut numeric important precum `GrLivArea`.
- **Preprocesarea simplificata:** `GarageYrBlt` lipsa este inlocuit cu mediana (desi inseamna „fara garaj”), iar selectia automata pentru log nu a inclus suprafetele principale.
- **Comparatie incompleta a preprocesarilor:** L2 si log au fost testate doar la 95%, iar combinatia **log + L2** nu a fost evaluata; varianta completa a laboratorului arata ca acolo logaritmarea schimba mult rezultatul.
- **Vizualizarea 2D** foloseste doar 2 din 45–135 de componente si poate ascunde suprapuneri sau separari din celelalte dimensiuni.
- **Casele noi** sunt patru exemple inventate, nu un set de test; ele verifica intuitia, nu generalizarea.

Rezumatul numeric de mai jos este generat automat din rezultate.

In [ ]:
at95 = summary[summary.Prag == .95]
best = at95.loc[at95.Silhouette.idxmax()]
base_rows = summary[summary.Metoda == "minmax"]
display(Markdown(
    f"La min-max, trecerea de la 98% la 80% reduce numarul de componente "
    f"de la **{int(base_rows.iloc[0].Componente)}** la **{int(base_rows.iloc[-1].Componente)}**. "
    f"La pragul de 95%, cel mai mare silhouette dintre variantele evaluate este "
    f"**{best.Silhouette:.3f}**, pentru **{best.Metoda}**, cu **k={int(best.k)}**. "
    "Aceasta este cea mai buna separare geometrica masurata aici, nu o dovada ca metoda "
    "descrie cel mai bine casele. Verificarea profilurilor ramane necesara."))
